# Session 3: Building a MIDAS model

This notebook is a guided lab. The goal is not just to **run** a MIDAS regression, but to understand how it is engineered, why the lag weights matter, and how it compares with simpler forecasting rules.

## Learning goals

By the end of the notebook you should be able to:

1. engineer a MIDAS design matrix from monthly industrial production,
2. understand and plot Beta weighting schemes,
3. estimate a restricted Beta-MIDAS model,
4. add an autoregressive quarterly term and build an ADL-MIDAS model,
5. compare AR, bridge, and MIDAS forecasts out of sample.

Throughout the notebook we work with **Germany (`DEU`)** and **quarterly GDP YoY growth** as the target.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

from MIDAS import (
    PALETTE,
    BetaMIDASRegressor,
    beta_weights,
    make_lags,
    rmse,
    mae,
    rolling_beta_midas_forecast,
    stack_midas_features,
    use_aer_style,
)

warnings.filterwarnings('ignore')
use_aer_style()

COUNTRY = 'DEU'
TARGET = 'gdp_yoy'
N_LAGS = 12
END_MONTH = 3          # try 1 or 2 later to mimic incomplete-quarter nowcasts
EVAL_START = '2005-01-01'
CRISIS_WINDOWS = [('2008-01-01', '2010-12-31'), ('2020-01-01', '2021-12-31')]


## 1. Load Germany’s monthly and quarterly panel

We will use:

- **quarterly real GDP YoY growth** as the target,
- **monthly industrial production YoY** as the high-frequency predictor,
- and later **lagged GDP growth** as a quarterly autoregressive control.

A useful mindset is:

- the **monthly panel** contains the information we would see during the quarter,
- the **quarterly panel** contains the target we want to explain or forecast.


In [ ]:
q = pd.read_csv('data/deu_quarterly.csv', parse_dates=['date'])
m = pd.read_csv('data/deu_monthly.csv', parse_dates=['date'])

q = q[q['country'] == COUNTRY].sort_values('date').reset_index(drop=True)
m = m[m['country'] == COUNTRY].sort_values('date').reset_index(drop=True)

print('Quarterly sample:', q['date'].min().date(), 'to', q['date'].max().date(), '| rows =', len(q))
print('Monthly sample:  ', m['date'].min().date(), 'to', m['date'].max().date(), '| rows =', len(m))

q[['date', 'gdp_yoy', 'ip_yoy', 'cpi_yoy', 'unemp', 'rate3m']].dropna().head(8)


## 2. First visual check

Before estimating anything, look at the data. GDP growth is quarterly and smoother; industrial production is monthly and noisier, but it often turns earlier.

That is exactly why MIDAS can be useful: it lets us exploit the **shape** of recent monthly information instead of collapsing everything into one crude quarterly average.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.4, 5.6), sharex=True)

axes[0].plot(q['date'], q['gdp_yoy'], color=PALETTE['myblue'], lw=1.6)
axes[0].axhline(0, color='#666666', lw=0.7)
axes[0].set_ylabel('GDP YoY (%)')
axes[0].set_title('Germany: quarterly GDP growth and monthly industrial production')

axes[1].plot(m['date'], m['ip_yoy'], color=PALETTE['myred'], lw=1.2)
axes[1].axhline(0, color='#666666', lw=0.7)
axes[1].set_ylabel('IP YoY (%)')
axes[1].set_xlabel('Date')

fig.tight_layout()
plt.show()


## 3. What is the MIDAS engineering problem?

Suppose the quarterly target is $y_t$ and the monthly indicator is $x_{t-k/3}$.

A simple restricted Beta-MIDAS regression is:

$$
 y_t = \alpha + \beta \sum_{k=0}^{K-1} w_k(\theta_1, \theta_2) x_{t-k/3} + \varepsilon_t,
$$

where the weights $w_k(\theta_1, \theta_2)$ come from a Beta polynomial and are constrained to sum to one.

The key idea is that the model **learns how much each monthly lag matters**.

An ADL-MIDAS extension adds quarterly persistence:

$$
 y_t = \alpha + \beta \sum_{k=0}^{K-1} w_k(\theta_1, \theta_2) x_{t-k/3} + \gamma y_{t-1} + \varepsilon_t.
$$

That extra $y_{t-1}$ term often stabilizes forecasts when GDP is persistent.


## 4. Stack the monthly lags that MIDAS needs

The code below transforms one monthly series into a quarterly design matrix.

- `L0` is the **latest available month** in the quarter,
- `L1` is the month before that,
- ...
- `L11` is the oldest monthly lag in the 12-month window.

This is the moment where the mixed-frequency problem becomes a standard regression problem.


In [ ]:
base = q[['date', 'country', 'gdp_yoy']].copy()
base = make_lags(base, ['gdp_yoy'], [1], by=None)

monthly_ip = m[['date', 'ip_yoy']].dropna().set_index('date').sort_index()
ip_stack = stack_midas_features(monthly_ip, pd.DatetimeIndex(base['date']), ['ip_yoy'], n_lags=N_LAGS, end_month=END_MONTH)

frame_ip = pd.concat([base.reset_index(drop=True), ip_stack.reset_index(drop=True)], axis=1)
frame_ip.head()


In [ ]:
example_q = pd.Timestamp('2008-07-01')
anchor = example_q.to_period('Q').end_time.normalize().replace(day=1)
months = pd.date_range(end=anchor, periods=N_LAGS, freq='MS')[::-1]
engineering = pd.DataFrame({
    'lag': [f'L{i}' for i in range(N_LAGS)],
    'month_used': months.strftime('%Y-%m'),
    'ip_yoy_value': frame_ip.loc[frame_ip['date'] == example_q, [f'ip_yoy_L{i}' for i in range(N_LAGS)]].iloc[0].values,
})
engineering


### Stop and think

For `END_MONTH = 3`, `L0` uses the last month of the quarter. If you changed `END_MONTH = 1`, you would only let the model see the **first** month of the quarter, which is closer to a real-time early nowcast.


## 5. Beta weights: why not estimate 12 free coefficients?

If we let every monthly lag have its own unrestricted coefficient, the quarterly sample becomes very thin very quickly.

Restricted MIDAS uses a smooth weighting function instead:

- **recent-loading** patterns put most mass on `L0`, `L1`, `L2`,
- **uniform** patterns treat all recent months similarly,
- **hump-shaped** patterns can emphasize the middle of the lag window.

That is the parsimony advantage of MIDAS.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.8))
ks = np.arange(1, N_LAGS + 1)
configs = [
    (1.0, 5.0, PALETTE['myblue'],  'Recent-loading'),
    (1.0, 1.0, PALETTE['pastgray'],'Uniform'),
    (2.0, 5.0, PALETTE['mygreen'], 'Hump-shaped'),
    (1.0, 2.0, PALETTE['myred'],   'Mild recency'),
]
for t1, t2, color, label in configs:
    ax.plot(ks, beta_weights(t1, t2, N_LAGS), marker='o', lw=1.4, color=color, label=label)
ax.set_xlabel('Monthly lag k (1 = latest month)')
ax.set_ylabel('Weight')
ax.set_title('Different Beta weighting patterns over 12 monthly lags')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()


## 6. Estimate a simple Beta-MIDAS model with industrial production only

This is the cleanest entry point:

- one monthly predictor: `ip_yoy`,
- no quarterly AR term yet,
- 12 monthly lags,
- weights learned from the data.


In [ ]:
midas_ip = BetaMIDASRegressor(monthly_vars=['ip_yoy'], n_lags=N_LAGS, low_freq_features=[])
midas_ip.fit_frame(frame_ip, target=TARGET)

print('Optimization success:', midas_ip.result_.success)
print('Message:', midas_ip.result_.message)

midas_ip.parameter_table()


In [ ]:
weights_ip = midas_ip.weight_frame()
weights_ip


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.6))
sub = weights_ip[weights_ip['variable'] == 'ip_yoy'].copy()
sub['lag_from_latest'] = sub['lag'] - 1
ax.plot(sub['lag_from_latest'], sub['weight'], marker='o', color=PALETTE['myblue'], lw=1.6)
ax.set_xlabel('Lag from latest month (0 = latest month)')
ax.set_ylabel('Estimated weight')
ax.set_title('Estimated Beta-MIDAS weights for German industrial production')
fig.tight_layout()
plt.show()


### Interpretation prompt

If the estimated weights are concentrated on the first few lags, the model is telling you that **very recent industrial production data matter most** for quarterly GDP growth.

If the weights are flatter, it means the model is effectively averaging over a broader monthly window.


## 7. Add a quarterly AR term: ADL-MIDAS

Now we enrich the model with one lag of quarterly GDP growth.

This matters because GDP growth is persistent. In practice, a simple autoregressive term often absorbs slow-moving dynamics, while the monthly predictor helps at turning points.


In [ ]:
adl_ip = BetaMIDASRegressor(monthly_vars=['ip_yoy'], n_lags=N_LAGS, low_freq_features=['gdp_yoy_l1'])
adl_ip.fit_frame(frame_ip, target=TARGET)
adl_ip.parameter_table()


## 8. Build out-of-sample forecasts: AR, bridge, MIDAS, ADL-MIDAS

We now compare four models on the **same Germany evaluation window**.

1. **AR(1)**: only lagged GDP growth.
2. **Bridge (IP quarterly average)**: a simple linear model using the quarterly average of monthly IP plus GDP persistence.
3. **Beta-MIDAS (IP)**: same monthly information, but with a learned weight curve.
4. **ADL-MIDAS (IP + AR)**: MIDAS plus lagged GDP growth.

The key principle is that all models should use the **same target** and be judged on a **common out-of-sample period**.


In [ ]:
def rolling_ols_forecast(frame, target, features, eval_start='2005-01-01', min_train=24):
    use = frame[['date', target] + features].dropna().copy().sort_values('date')
    rows = []
    for dt in sorted(use.loc[use['date'] >= pd.Timestamp(eval_start), 'date'].unique()):
        train = use[use['date'] < dt]
        test = use[use['date'] == dt]
        if len(train) < min_train or test.empty:
            continue
        model = LinearRegression().fit(train[features], train[target])
        out = test[['date', target]].copy()
        out['y_hat'] = model.predict(test[features])
        rows.append(out)
    return pd.concat(rows, ignore_index=True).rename(columns={target: 'y_true'})

quarterly_baselines = q[['date', 'gdp_yoy', 'ip_yoy']].copy()
quarterly_baselines = make_lags(quarterly_baselines, ['gdp_yoy'], [1], by=None)
quarterly_baselines['ip_yoy_qavg_l0'] = q['ip_yoy'].values

fc_ar1 = rolling_ols_forecast(quarterly_baselines, 'gdp_yoy', ['gdp_yoy_l1'], eval_start=EVAL_START)
fc_bridge = rolling_ols_forecast(quarterly_baselines, 'gdp_yoy', ['gdp_yoy_l1', 'ip_yoy_qavg_l0'], eval_start=EVAL_START)
fc_midas = rolling_beta_midas_forecast(frame_ip, 'gdp_yoy', ['ip_yoy'], n_lags=N_LAGS, low_freq_features=[], eval_start=EVAL_START, min_train=24)
fc_adl = rolling_beta_midas_forecast(frame_ip, 'gdp_yoy', ['ip_yoy'], n_lags=N_LAGS, low_freq_features=['gdp_yoy_l1'], eval_start=EVAL_START, min_train=24)

forecast_dict = {
    'AR(1)': fc_ar1,
    'Bridge (IP avg)': fc_bridge,
    'Beta-MIDAS (IP)': fc_midas,
    'ADL-MIDAS (IP + AR)': fc_adl,
}


In [ ]:
common = None
for name, df in forecast_dict.items():
    sub = df[['date', 'y_true', 'y_hat']].rename(columns={'y_hat': name})
    if common is None:
        common = sub.copy()
    else:
        common = common.merge(sub[['date', name]], on='date', how='inner')

common = common.rename(columns={'y_true': 'actual'})
metrics = []
for name in forecast_dict:
    metrics.append({'model': name, 'RMSE': rmse(common['actual'], common[name]), 'MAE': mae(common['actual'], common[name])})
metrics = pd.DataFrame(metrics).sort_values('RMSE').reset_index(drop=True)
metrics


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
colors = [PALETTE['mygreen'] if 'MIDAS' in m else PALETTE['myblue'] for m in metrics['model']]
ax.bar(metrics['model'], metrics['RMSE'], color=colors)
ax.set_ylabel('RMSE (pp)')
ax.set_title('Germany: out-of-sample forecast accuracy')
ax.grid(True, axis='y')
ax.set_axisbelow(True)
plt.xticks(rotation=12, ha='right')
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.6, 6.0), sharex=False)
plot_specs = [
    ('2006-01-01', '2012-12-31', 'Global Financial Crisis window'),
    ('2018-01-01', '2022-12-31', 'COVID window'),
]
colors = {
    'actual': PALETTE['pastgray'],
    'AR(1)': PALETTE['myblue'],
    'Bridge (IP avg)': '#5f7aa6',
    'Beta-MIDAS (IP)': PALETTE['myred'],
    'ADL-MIDAS (IP + AR)': PALETTE['mygreen'],
}
for ax, (start, end, title) in zip(axes, plot_specs):
    sub = common[(common['date'] >= pd.Timestamp(start)) & (common['date'] <= pd.Timestamp(end))].copy()
    ax.plot(sub['date'], sub['actual'], color=colors['actual'], lw=1.8, label='Actual GDP YoY')
    for name in forecast_dict:
        ax.plot(sub['date'], sub[name], lw=1.4, label=name, color=colors[name])
    ax.axhline(0, color='#666666', lw=0.7)
    ax.set_title(title)
    ax.set_ylabel('YoY growth (%)')
    ax.grid(True)
axes[0].legend(ncol=2, frameon=False, loc='lower right')
axes[-1].set_xlabel('Quarter')
fig.tight_layout()
plt.show()


## 9. What should you conclude?

On Germany, the usual pattern is that:

- **AR(1)** is simple and often not terrible,
- **Bridge** already improves a lot by injecting current-quarter industrial production,
- **Beta-MIDAS** improves further because it uses the monthly profile more intelligently,
- **ADL-MIDAS** can help if GDP persistence matters enough in the evaluation period.

The important lesson is not that MIDAS always wins mechanically. It is that MIDAS gives you a disciplined way to use monthly information **without exploding the number of parameters**.
